# Provenance and framing diagrams

Figures for the introduction (`fig:pillars`) and the data-infrastructure section
(`fig:prov-chain`, `fig:prov-roles`).

These are schematics, not plots: nothing here reads simulation data. Node types,
collection roles and edge directions follow `aiida-pino/database.md` and the code
it describes, so the diagram and the store stay in step. The docstring below
records each fact and where it was checked.

Source data: none. Regeneration is guarded by `figstyle.REPLACE_FIGURES`
(False by default).

In [ ]:
"""Draft diagrams for sec:pipeline and the introduction.

Every fact drawn here is taken from ../aiida-pino/database.md ("Node graph",
"Provenance capture", "Node types") and cross-checked against the code, not from
memory. The load-bearing ones:

  - the import job (RawFlow3DToDatasetCalculation) create-links BOTH the
    simulations and the imported collection; the solver does not create nodes
  - the case declaration is a stored orm.Dict input of the run job
    (calculations/flow3d_run.py: spec.input("run_config", valid_type=orm.Dict)),
    so it is inside the store, not an external file
  - simulations --inputs--> compose_dataset --create--> training dataset
  - dataset --input--> PinoTrainCalculation --create--> artifacts
                            --> materialize_model_node --create--> model
  - model + dataset --inputs--> materialize_evaluation_node --create-->
    evaluation, which holds AGGREGATE metrics PER OUTPUT FIELD over the
    evaluation set, not one metric per simulation
  - process nodes own repository content too (uploaded inputs, retrieved
    outputs), so "unfilled = metadata only" holds for the blue nodes only
  - membership is a REAL edge only for nodes created from 2026-07-30 on; older
    collections carry members as a stored `member_simulation_uuids` attribute
  - roles (dataset_hierarchy.py): a simulation belongs to AT MOST ONE base
    dataset -- create_base_dataset raises SimulationAlreadyAttachedError
    otherwise, so base collections are disjoint; a subset selects members of one
    base; a *composed* collection is built from datasets TAKEN WHOLE, never from
    raw simulations (breaking sources up is the separate *custom* kind); a
    derived collection stores a transformation profile of its parent and keeps
    the parent's members, recomputing arrays on read
"""
import sys
import pathlib

sys.path.insert(0, str(pathlib.Path.cwd()))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import figstyle

figstyle.apply()

NODE = figstyle.PALETTE[0]    # metadata-only node (PostgreSQL row only)
PROC = figstyle.PALETTE[1]    # process: CalcJob or calcfunction
REPO = figstyle.PALETTE[2]    # node holding the array payload
INK = "#1a1a1a"
MUTE = "#5a5a5a"
FAINT = "#9A9A9A"

# Box text layout, shared by all three diagrams (see box()).
TOP_INSET = 0.45      # title inset below the top edge, in title-font heights
SUBSPACING = 1.15     # line spacing of the italic sub-label

BODY = figstyle.DIAGRAM_BODY
SMALL = figstyle.DIAGRAM_SMALL
TINY = figstyle.DIAGRAM_TINY

Shared drawing helpers. All sizes and colours come from `figstyle`.

In [ ]:
def box(ax, x, y, w, h, label, sub=None, ec=INK, lw=1.5, ls="-", repo=False,
        fs=None):
    """Title over an optional italic sub-label, both kept clear of the border.

    The sub block is laid out from its line count, so a two-line sub cannot
    overrun the bottom edge the way a fixed offset does.
    """
    fc = ec + "1A" if repo else "white"
    ax.add_patch(FancyBboxPatch((x, y), w, h,
                                boxstyle="round,pad=0.012,rounding_size=0.02",
                                ec=ec, fc=fc, lw=lw, linestyle=ls, zorder=2))
    cx = x + w / 2
    tfs = fs or BODY
    if not sub:
        ax.text(cx, y + h / 2, label, ha="center", va="center",
                fontsize=tfs, color=INK, zorder=3)
        return
    # data units per point, so the text block is laid out on its true height
    y0, y1 = ax.get_ylim()
    upp = (y1 - y0) / (ax.get_position().height
                       * ax.figure.get_size_inches()[1] * 72.0)
    nl = sub.count("\n") + 1
    t_h = tfs * upp
    s_h = nl * SMALL * SUBSPACING * upp
    gap = 0.30 * SMALL * upp
    # The title sits a FIXED distance below the top edge, so every box in a
    # figure carries its title in the same relative place whatever its height
    # or its number of sub lines. Centring the whole block instead tied the
    # title's position to the sub's line count, which put the short evaluation
    # box of fig:prov-chain and the role boxes of fig:prov-roles almost on
    # their own border. Clamped so a tall sub still clears the bottom edge.
    inset = min(TOP_INSET * t_h, max(0.0, h - (t_h + gap + s_h) - 0.25 * t_h))
    top = y + h - inset
    ax.text(cx, top, label, ha="center", va="top", fontsize=tfs, color=INK,
            zorder=3)
    ax.text(cx, top - t_h - gap, sub, ha="center", va="top", fontsize=SMALL,
            color=MUTE, style="italic", linespacing=SUBSPACING, zorder=3)


def link(ax, p, q, dashed=False, rad=0.0, lw=1.4, color=INK, zorder=1):
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=13,
                                 lw=lw, color=color, zorder=zorder,
                                 linestyle=(0, (4, 3)) if dashed else "-",
                                 connectionstyle=f"arc3,rad={rad}",
                                 shrinkA=3, shrinkB=3))


DASH = (0, (1, 2.2))          # derived from (dotted: must not read as DASHDOT)
DASHDOT = (0, (5, 2, 1, 2))   # built from, whole


def route(ax, pts, color=INK, lw=1.2, dashed=False, head=True, zorder=4,
          ls=None):
    """Orthogonal connector, arrowhead on the final segment."""
    style = ls if ls is not None else (DASH if dashed else "-")
    xs, ys = zip(*pts)
    if head:
        xs, ys = xs[:-1], ys[:-1]
    ax.plot(xs, ys, color=color, lw=lw, zorder=zorder, linestyle=style,
            solid_capstyle="round", solid_joinstyle="round")
    if head:
        ax.add_patch(FancyArrowPatch(pts[-2], pts[-1], arrowstyle="-|>",
                                     mutation_scale=12, lw=lw, color=color,
                                     linestyle=style,
                                     zorder=zorder, shrinkA=0, shrinkB=2))


def canvas(units_w=13.6, units_h=7.0):
    """A diagram canvas: one axes filling the figure, equal scale in x and y.

    Width is the house width for a \\textwidth figure, so the diagram prints at
    REDUCTION like every other figure and its text at figstyle.FONT*REDUCTION.
    One data unit is the same length in x and y, so box insets and the measured
    text widths below are exact instead of estimates.
    """
    w = figstyle.width_in(1.0)
    fig = plt.figure(figsize=(w, w * units_h / units_w), constrained_layout=False)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, units_w); ax.set_ylim(0, units_h); ax.axis("off")
    return fig, ax


def text_width(ax, s, **kw):
    """Width of a string in data units at the house font."""
    t = ax.text(0, 0, s, fontsize=BODY, **kw)
    r = ax.figure.canvas.get_renderer()
    bb = t.get_window_extent(r).transformed(ax.transData.inverted())
    t.remove()
    return bb.width


def legend_row(ax, entries, y, x0=0.25, gap=0.7):
    """Legend entries left to right, each placed after the measured end of the last.

    `entries` holds (draw_handle, label); draw_handle(ax, x, y) draws a handle
    0.85 units wide. Fixed slots let a long label run into the next handle.
    """
    x = x0
    for draw, label in entries:
        draw(ax, x, y)
        ax.text(x + 1.05, y, label, fontsize=TINY, va="center", color=INK)
        x += 1.05 + text_width(ax, label) + gap


## Generation and provenance chain

One case from declaration to reported metric. The case declaration is a stored
`orm.Dict` input of the run job, so it sits inside the store like everything
else.

In [ ]:
def panel_a():
    """Generation and provenance chain: 3 columns, 4 rows, one flow."""
    fig, ax = canvas(13.6, 10.9)
    ax.set_ylim(-0.75, 10.15)

    W, H = 4.00, 1.75
    CX = [0.50, 4.80, 9.10]
    RY = [7.50, 5.25, 3.00, 0.75]

    ax.add_patch(FancyBboxPatch((0.15, 0.40), 13.30, 9.55,
                                boxstyle="round,pad=0.02,rounding_size=0.05",
                                ec="#B0B0B0", fc="#FBFBFB", lw=1.2, zorder=0))
    ax.text(13.28, 9.62, "AiiDA provenance database", fontsize=TINY,
            color="#5E5E5E", style="italic", ha="right", va="center", zorder=3)

    box(ax, CX[0], RY[0], W, H, "Simulation declaration",
        sub="Parameters, geometry,\npath, material", ec=NODE)
    box(ax, CX[1], RY[0], W, H, "Run job",
        sub="Renders the deck,\nruns FLOW-3D WELD", ec=PROC)
    box(ax, CX[2], RY[0], W, H, "Import job",
        sub="Reads the arrays\nand the manifest", ec=PROC)
    link(ax, (CX[0] + W, RY[0] + H / 2), (CX[1], RY[0] + H / 2))
    link(ax, (CX[1] + W, RY[0] + H / 2), (CX[2], RY[0] + H / 2))

    box(ax, CX[0], RY[1], W, H, "Dataset",
        sub="Ordered member list,\nno arrays", ec=NODE)
    box(ax, CX[1], RY[1], W, H, "Compose", sub="Calcfunction", ec=PROC)
    box(ax, CX[2], RY[1], W, H, "Simulation",
        sub="Fields over time,\ncopies its manifest", ec=REPO, repo=True)
    link(ax, (CX[2] + W / 2, RY[0]), (CX[2] + W / 2, RY[1] + H))
    link(ax, (CX[2], RY[1] + H / 2), (CX[1] + W, RY[1] + H / 2))
    link(ax, (CX[1], RY[1] + H / 2), (CX[0] + W, RY[1] + H / 2))

    box(ax, CX[0], RY[2], W, H, "Train job",
        sub="Training calculation", ec=PROC)
    box(ax, CX[1], RY[2], W, H, "Model",
        sub="Checkpoints,\nconfiguration", ec=REPO, repo=True)
    box(ax, CX[2], RY[2], W, H, "Evaluate", sub="Calcfunction", ec=PROC)
    link(ax, (CX[0] + W / 2, RY[1]), (CX[0] + W / 2, RY[2] + H))
    link(ax, (CX[0] + W, RY[2] + H / 2), (CX[1], RY[2] + H / 2))
    link(ax, (CX[1] + W, RY[2] + H / 2), (CX[2], RY[2] + H / 2))

    box(ax, CX[2], RY[3], W, H, "Evaluation",
        sub="Aggregate metrics\nper output field", ec=NODE)
    link(ax, (CX[2] + W - 1.05, RY[2]), (CX[2] + W - 1.05, RY[3] + H))

    # the held-out dataset is the evaluate step's second input; the lane runs in
    # the gap between the dataset row and the train row
    lane = (RY[1] + RY[2] + H) / 2
    route(ax, [(CX[0] + 2.70, RY[1]), (CX[0] + 2.70, lane), (10.00, lane),
               (10.00, RY[2] + H)], lw=1.2, zorder=1)

    def swatch(c, rp):
        def draw(ax, x, y):
            ax.add_patch(FancyBboxPatch((x, y - 0.21), 0.85, 0.42,
                                        boxstyle="round,pad=0.012,rounding_size=0.02",
                                        ec=c, fc=c + "1A" if rp else "white", lw=1.5))
        return draw
    legend_row(ax, [(swatch(REPO, True), "Holds the field arrays"),
                    (swatch(NODE, False), "Metadata only"),
                    (swatch(PROC, False), "Process")], y=-0.30, x0=0.35, gap=0.9)

    figstyle.save(fig, "provenance_chain")
    return fig

fig = panel_a()


## Collection roles

Base collections are disjoint, because a simulation belongs to at most one base
dataset. A subset picks simulations out of one base; a composed collection takes
whole datasets as members; a derived collection stores a transformation of its
parent.

In [ ]:
def panel_b():
    """Collection roles. Base collections are disjoint; composed takes datasets
    whole; a subset picks simulations out of one base."""
    fig, ax = canvas(13.6, 7.6)
    ax.set_ylim(-1.15, 6.45)

    SX, PITCH, SW = 2.30, 1.72, 1.12
    sim_x = [SX + PITCH * i for i in range(6)]
    for i, x in enumerate(sim_x):
        box(ax, x, 0.55, SW, 0.66, f"{i + 1}", ec=REPO, lw=1.4, repo=True)
    ax.text(SX, 0.22, "Simulations, the only nodes holding arrays",
            fontsize=TINY, color=MUTE, style="italic", ha="left", va="top")

    AX0, AW = 2.20, 4.76
    BX0 = sim_x[3] - 0.10
    BH, BY = 1.25, 2.35
    BTOP = BY + BH
    box(ax, AX0, BY, AW, BH, "Base A", sub="One simulation batch", ec=NODE)
    box(ax, BX0, BY, AW, BH, "Base B", sub="A second simulation batch", ec=NODE)

    # the composed box carries the longest line, so it is the widest of the row
    RY, RH = 4.55, 1.75
    SUBX, SUBW = 0.15, 3.90
    COMX, COMW = 4.45, 4.50
    DERX, DERW = 9.35, 3.90
    box(ax, SUBX, RY, SUBW, RH, "Subset",
        sub="Picks from one base,\na stored split", ec=NODE)
    box(ax, COMX, RY, COMW, RH, "Composed",
        sub="Built from A and B,\nwith all their members", ec=NODE)
    box(ax, DERX, RY, DERW, RH, "Derived",
        sub="A transformation of B,\nrecomputed on read", ec=NODE)

    # base membership: each simulation belongs to exactly one base
    for x in sim_x[:3]:
        link(ax, (x + SW / 2, 1.21), (AX0 + AW / 2, BY), color=FAINT, lw=1.1)
    for x in sim_x[3:]:
        link(ax, (x + SW / 2, 1.21), (BX0 + AW / 2, BY), color=FAINT, lw=1.1)

    # subset references two simulations directly: one shared lane, left of A
    for x in sim_x[:2]:
        ax.plot([x + SW / 2, x + SW / 2, 1.15], [1.21, 1.78, 1.78],
                color=INK, lw=1.2, zorder=4, solid_capstyle="round",
                solid_joinstyle="round")
    jog = (BTOP + RY) / 2
    route(ax, [(1.15, 1.78), (1.15, jog), (2.50, jog), (2.50, RY)], lw=1.2)

    # composed references the two base collections, not their simulations
    route(ax, [(AX0 + AW - 0.80, BTOP), (AX0 + AW - 0.80, jog),
               (6.45, jog), (6.45, RY)], lw=1.2, ls=DASHDOT)
    route(ax, [(BX0 + 0.80, BTOP), (BX0 + 0.80, jog),
               (6.95, jog), (6.95, RY)], lw=1.2, ls=DASHDOT)

    # derived records the parent it transforms and keeps that parent's members
    route(ax, [(BX0 + AW - 0.80, BTOP), (BX0 + AW - 0.80, RY)],
          lw=1.2, dashed=True)
    # a subset records the base it was drawn from
    route(ax, [(AX0 + 1.20, BTOP), (AX0 + 1.20, RY)], lw=1.2, dashed=True)

    def stroke(st):
        def draw(ax, x, y):
            ax.plot([x, x + 0.85], [y, y], color=INK, lw=1.2, linestyle=st)
            ax.add_patch(FancyArrowPatch((x + 0.55, y), (x + 0.95, y),
                                         arrowstyle="-|>", mutation_scale=12,
                                         lw=1.2, color=INK, shrinkA=0, shrinkB=0,
                                         linestyle=st))
        return draw
    legend_row(ax, [(stroke("-"), "References its members"),
                    (stroke(DASHDOT), "Built from, whole"),
                    (stroke(DASH), "Derived from")], y=-0.80, x0=0.15, gap=0.55)

    figstyle.save(fig, "collection_roles")
    return fig

fig = panel_b()


## Three pillars

The framing figure for the introduction.

In [ ]:
def panel_pillars():
    """Three pillars, carrying the section each is settled in."""
    fig, ax = canvas(13.6, 9.05)

    TOPX, TOPW = 1.60, 10.40
    box(ax, TOPX, 7.90, TOPW, 1.00,
        "Universal ML surrogate for laser melting as a process", ec=INK, lw=1.6)

    # one colour per pillar, carried by its border and its section label.
    # Figs 2 and 3 give these same hues a node meaning, but they each carry
    # their own legend and this figure has no nodes, so nothing is inherited.
    # Items wrap to the width a third of the page holds at the house text size.
    PW, PY, PH = 4.30, 2.05, 5.20
    LINE, GAP = 0.42, 0.22
    pillars = [
        (0.15, "Data infrastructure", "Section 2.1", PROC,
         ["Organized and\nsearchable data", "Efficient data storage",
          "Provenance and\nreproducibility"]),
        (4.65, "Generalization", "Section 3.2", NODE,
         ["Weld path", "Process parameters", "Material", "Melting regime"]),
        (9.15, "Parameter-space\ncoverage", "Section 3.3", REPO,
         ["Self-extending\ndata generation", "Efficient coverage",
          "Which data to\noptimally reduce\nmodel error"]),
    ]
    top = PY + PH
    for x, title, sec, col, items in pillars:
        cx = x + PW / 2
        box(ax, x, PY, PW, PH, "", ec=col, lw=1.6)
        # titles hang from the same line; the section label sits below a
        # two-line title slot, so all three line up whatever their length
        ax.text(cx, top - 0.22, title, ha="center", va="top", fontsize=BODY,
                color=INK, zorder=3, linespacing=1.1)
        ax.text(cx, top - 0.22 - 2 * LINE - 0.05, sec, ha="center", va="top",
                fontsize=TINY, color=col, style="italic", zorder=3)
        yy = top - 0.22 - 3 * LINE - 0.30
        for it in items:
            ax.text(cx, yy, it, ha="center", va="top", fontsize=SMALL,
                    color=MUTE, zorder=3, linespacing=1.15)
            yy -= LINE * (1 + it.count("\n")) + GAP
        # aim every arrow at the top box; the outer two lean inward onto it
        lean = {0.15: 0.9, 9.15: -0.9}.get(x, 0.0)
        link(ax, (cx, top), (cx + lean, 7.90), lw=1.3)
        link(ax, (cx, 1.40), (cx, PY), lw=1.3)

    box(ax, 0.15, 0.15, 13.30, 1.25, "Validated solver and generated corpus",
        sub="Benchmark-anchored physics, Supplementary Note B",
        ec=INK, lw=1.6)

    figstyle.save(fig, "three_pillars")
    return fig

def check_section_numbers():
    """Fig 1 hardcodes section numbers into a PDF; nothing else catches a drift.

    Reads the last build's .aux and fails loudly if a pillar now names the wrong
    section. Silent when the manuscript has not been built yet.
    """
    import re
    aux = pathlib.Path.cwd().parent / "rendering" / "main.aux"
    if not aux.exists():
        print("[pillars] no rendering/main.aux yet; section numbers unchecked")
        return
    text = aux.read_text()
    wrong = []
    for label, shown in [("sec:infra", "Section 2.1"),
                         ("sec:axes", "Section 3.2"),
                         ("sec:al-results", "Section 3.3"),
                         ("sec:calibration", "Supplementary Note B")]:
        m = re.search(r"\\newlabel\{" + re.escape(label) + r"\}\{\{([^}]*)\}", text)
        expect = shown.replace("Supplementary Note ", "").replace("Section ", "")
        if m and m.group(1) != expect:
            wrong.append(f"{label}: figure says {shown}, manuscript says {m.group(1)}")
    if wrong:
        raise AssertionError("three_pillars is stale -- " + "; ".join(wrong))
    print("[pillars] section numbers match the manuscript")


check_section_numbers()
fig = panel_pillars()
